# Teste de viabilidade: CODE-15% no Colab

Notebook **descartável**, só para medir tempo: download, leitura e treino de uma CNN 1D pequena com GPU, usando **uma parte** da CODE-15% (~20 mil ECGs).

**Como rodar:** Ambiente de execução → Alterar o tipo de ambiente de execução → **GPU (T4)** → Salvar. Depois, Ambiente de execução → **Executar tudo**. Leva uns 15-30 min, a maior parte no download.

No fim, a última célula mostra um resumo dos tempos. Mande esse resumo para o Claude.

## 1. GPU disponível?

In [ ]:
import time, os
import tensorflow as tf
tempos = {}
print("GPUs:", tf.config.list_physical_devices("GPU"))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Download (tabela + parte 0)

Baixa em 16 conexões paralelas, porque o Zenodo limita a velocidade de cada conexão. O script confere se todos os bytes chegaram.

In [ ]:
import urllib.request
from concurrent.futures import ThreadPoolExecutor


def baixar(url, destino, n=16):
    """Baixa em n pedaços paralelos (HTTP Range), retomando conexões que caem."""
    with urllib.request.urlopen(urllib.request.Request(url, headers={"Range": "bytes=0-0"})) as r:
        total = int(r.headers["Content-Range"].split("/")[1])
    with open(destino, "wb") as f:
        f.truncate(total)
    passo = -(-total // n)
    baixado = [0] * n

    def pedaco(i):
        ini, fim = i * passo, min(total, (i + 1) * passo) - 1
        for _ in range(100):
            if baixado[i] >= fim - ini + 1:
                return
            try:
                pos = ini + baixado[i]
                req = urllib.request.Request(url, headers={"Range": f"bytes={pos}-{fim}"})
                with urllib.request.urlopen(req, timeout=60) as r, open(destino, "r+b") as f:
                    f.seek(pos)
                    while bloco := r.read(1 << 20):
                        f.write(bloco)
                        baixado[i] += len(bloco)
            except Exception as e:
                print(f"pedaço {i}: {e!r}, retomando")
                time.sleep(3)
        raise RuntimeError(f"pedaço {i} incompleto")

    with ThreadPoolExecutor(n) as ex:
        list(ex.map(pedaco, range(n)))
    assert sum(baixado) == total


BASE = "https://zenodo.org/records/4916206/files"
t0 = time.time()
urllib.request.urlretrieve(f"{BASE}/exams.csv?download=1", "exams.csv")
baixar(f"{BASE}/exams_part0.zip?download=1", "exams_part0.zip")
tempos["download (2,7 GB)"] = time.time() - t0
print(f"download: {tempos['download (2,7 GB)'] / 60:.1f} min")

## 3. Leitura

Descompacta, lê os sinais e junta com os rótulos. Exames todos zerados são descartados. Os sinais ficam em `float16` para caber na memória.

In [ ]:
import zipfile
import h5py
import numpy as np
import pandas as pd

t0 = time.time()
zipfile.ZipFile("exams_part0.zip").extractall(".")
with h5py.File("exams_part0.hdf5", "r") as f:
    X = f["tracings"][:].astype(np.float16)
    ids = f["exam_id"][:]
tempos["descompactar + ler"] = time.time() - t0

ROTULOS = ["1dAVb", "RBBB", "LBBB", "SB", "ST", "AF"]
tabela = pd.read_csv("exams.csv").set_index("exam_id")
manter = np.isin(ids, tabela.index) & (np.abs(X).max(axis=(1, 2)) > 0)
X, ids = X[manter], ids[manter]
info = tabela.loc[ids]
y = info[ROTULOS].values.astype(np.float32)
print(X.shape, "| pacientes:", info["patient_id"].nunique())

## 4. Split por paciente

80% dos pacientes para treino e 20% para validação. Nenhum paciente aparece nos dois.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

treino, val = next(GroupShuffleSplit(test_size=0.2, random_state=42).split(X, y, groups=info["patient_id"]))
print("treino:", len(treino), "| validação:", len(val),
      "| pacientes em comum:", len(set(info["patient_id"].iloc[treino]) & set(info["patient_id"].iloc[val])))

## 5. Treino de teste (2 épocas)

CNN 1D pequena, a mesma medida no computador da Renata (~400 mil parâmetros). Saída **sigmoid** com 6 neurônios, porque um ECG pode ter mais de uma alteração ao mesmo tempo (multi-rótulo).

O objetivo é **medir tempo**. O AUROC no fim é só para ver se o modelo está aprendendo alguma coisa; não é um resultado do projeto.

In [ ]:
tf.keras.utils.set_random_seed(42)


def criar_cnn():
    camadas = [tf.keras.layers.Input(shape=(4096, 12))]
    for filtros in (32, 64, 128):
        camadas += [tf.keras.layers.Conv1D(filtros, 15, padding="same", activation="relu"),
                    tf.keras.layers.MaxPooling1D(4)]
    camadas += [tf.keras.layers.Conv1D(128, 15, padding="same", activation="relu"),
                tf.keras.layers.GlobalAveragePooling1D(),
                tf.keras.layers.Dense(6, activation="sigmoid")]
    return tf.keras.Sequential(camadas)


modelo = criar_cnn()
modelo.compile(optimizer="adam", loss="binary_crossentropy")
tempos_epoca = []
for epoca in range(2):
    t0 = time.time()
    modelo.fit(X[treino], y[treino], batch_size=64, epochs=1, verbose=2)
    tempos_epoca.append(time.time() - t0)
tempos["época 1 (inclui compilação)"], tempos["época 2"] = tempos_epoca

In [ ]:
from sklearn.metrics import roc_auc_score

previsto = modelo.predict(X[val], batch_size=256, verbose=0)
for k, r in enumerate(ROTULOS):
    print(f"{r}: AUROC {roc_auc_score(y[val][:, k], previsto[:, k]):.2f} ({int(y[val][:, k].sum())} positivos)")

## 6. Resumo (mandar para o Claude)

In [ ]:
print("GPU:", tf.config.list_physical_devices("GPU"))
print(f"exames de treino: {len(treino)}")
for etapa, s in tempos.items():
    print(f"{etapa}: {s / 60:.1f} min ({s:.0f} s)")
por_exame = tempos["época 2"] / len(treino)
for n in (60_000, 250_000):
    print(f"estimativa de uma época com {n:,} exames: ~{por_exame * n / 60:.1f} min")